# Memory Agent

## 回顾

我们创建了一个聊天机器人，将语义记忆保存到单个[用户资料](https://docs.langchain.com/oss/python/concepts/memory#profile)或[集合](https://docs.langchain.com/oss/python/concepts/memory#collection)中。

我们介绍了 [Trustcall](https://github.com/hinthornw/trustcall) 作为更新任一 schema 的方式。

## 目标

现在，我们将整合我们所学的内容，构建一个具��长期记忆的 Agent。

我们的 Agent `task_mAIstro` 将帮助我们管理待办事项列表！

我们之前构建的聊天机器人*总是*反思对话并保存记忆。

`task_mAIstro` 将*决定何时*保存记忆（待办事项列表中的条目）。

我们之前构建的聊天机器人总是保存一种类型的记忆，资料或集合。

`task_mAIstro` 可以决定是保存到用户资料还是待办事项集合。

除了语义记忆，`task_mAIstro` 还将管理程序性记忆。

这允许用户更新他们创建待办事项的偏好。

In [ ]:
%%capture --no-stderr
%pip install -U langchain_deepseek langgraph trustcall langchain_core

In [ ]:
import os, getpass

def _set_env(var: str):
    # 检查变量是否已在 OS 环境变量中设置
    env_value = os.environ.get(var)
    if not env_value:
        # 如果未设置，提示用户输入
        env_value = getpass.getpass(f"{var}: ")
    
    # 为当前进程设置环境变量
    os.environ[var] = env_value

_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

In [ ]:
_set_env("DEEPSEEK_API_KEY")

## 查看 Trustcall 更新

Trustcall 创建和更新 JSON schema。

如果我们想要了解 Trustcall 所做的*具体更改*呢？

例如，我们之前看到 Trustcall 有一些自己的工具用于：

* 从验证失败中自我修正 -- [查看追踪示例](https://smith.langchain.com/public/5cd23009-3e05-4b00-99f0-c66ee3edd06e/r/9684db76-2003-443b-9aa2-9a9dbc5498b7)
* 更新现有文档 -- [查看追踪示例](https://smith.langchain.com/public/f45bdaf0-6963-4c19-8ec9-f4b7fe0f68ad/r/760f90e1-a5dc-48f1-8c34-79d6a3414ac3)

了解这些工具对我们即将构建的 agent 很有用。

下面我们将展示如何做到这一点！

In [ ]:
from pydantic import BaseModel, Field

class Memory(BaseModel):
    content: str = Field(description="The main content of the memory. For example: User expressed interest in learning about French.")

class MemoryCollection(BaseModel):
    memories: list[Memory] = Field(description="A list of memories about the user.")

我们可以向 Trustcall 提取器添加一个[监听器](https://python.langchain.com/docs/how_to/lcel_cheatsheet/#add-lifecycle-listeners) <!-- 链接已失效，但找不到更好的链接 -->。

这将把提取器执行的运行传递给我们将定义的 `Spy` 类。

我们的 `Spy` 类将提取 Trustcall 所做的工具调用的相关信息。

In [ ]:
from trustcall import create_extractor
from langchain_deepseek import ChatDeepSeek

# 检查 Trustcall 所做的工具调用
class Spy:
    def __init__(self):
        self.called_tools = []

    def __call__(self, run):
        # 收集提取器所做的工具调用信息。
        q = [run]
        while q:
            r = q.pop()
            if r.child_runs:
                q.extend(r.child_runs)
            if r.run_type == "chat_model":
                self.called_tools.append(
                    r.outputs["generations"][0][0]["message"]["kwargs"]["tool_calls"]
                )

# 初始化 spy
spy = Spy()

# 初始化模型
model = ChatDeepSeek(model="deepseek-chat", temperature=0)

# 创建提取器
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
)

# 将 spy 添加为监听器
trustcall_extractor_see_all_tool_calls = trustcall_extractor.with_listeners(on_end=spy)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 指令
instruction = """Extract memories from the following conversation:"""

# 对话
conversation = [HumanMessage(content="Hi, I'm Lance."), 
                AIMessage(content="Nice to meet you, Lance."), 
                HumanMessage(content="This morning I had a nice bike ride in San Francisco.")]

# 调用提取器
result = trustcall_extractor.invoke({"messages": [SystemMessage(content=instruction)] + conversation})

In [ ]:
# 消息包含工具调用
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# 响应包含符合 schema 的记忆
for m in result["responses"]: 
    print(m)

In [ ]:
# 元数据包含工具调用
for m in result["response_metadata"]: 
    print(m)

In [ ]:
# 更新对话
updated_conversation = [AIMessage(content="That's great, did you do after?"), 
                        HumanMessage(content="I went to Tartine and ate a croissant."),                        
                        AIMessage(content="What else is on your mind?"),
                        HumanMessage(content="I was thinking about my Japan, and going back this winter!"),]

# 更新指令
system_msg = """Update existing memories and create new ones based on the following conversation:"""

# 我们将保存现有记忆，赋予它们 ID、键（工具名称）和值
tool_name = "Memory"
existing_memories = [(str(i), tool_name, memory.model_dump()) for i, memory in enumerate(result["responses"])] if result["responses"] else None
existing_memories

In [ ]:
# 使用更新的对话和现有记忆调用提取器
result = trustcall_extractor_see_all_tool_calls.invoke({"messages": updated_conversation, 
                                                        "existing": existing_memories})

In [ ]:
# 元数据包含工具调用
for m in result["response_metadata"]: 
    print(m)

In [ ]:
# 消息包含工具调用
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# 解析后的响应
for m in result["responses"]:
    print(m)

In [ ]:
# 检查 Trustcall 所做的工具调用
spy.called_tools

In [ ]:
def extract_tool_info(tool_calls, schema_name="Memory"):
    """从工具调用中提取 patches 和新记忆的信息。
    
    参数：
        tool_calls：来自模型的工具调用列表
        schema_name：schema 工具的名称（例如，"Memory"、"ToDo"、"Profile"）
    """

    # 初始化更改列表
    changes = []
    
    for call_group in tool_calls:
        for call in call_group:
            if call['name'] == 'PatchDoc':
                changes.append({
                    'type': 'update',
                    'doc_id': call['args']['json_doc_id'],
                    'planned_edits': call['args']['planned_edits'],
                    'value': call['args']['patches'][0]['value']
                })
            elif call['name'] == schema_name:
                changes.append({
                    'type': 'new',
                    'value': call['args']
                })

    # 将结果格式化为单个字符串
    result_parts = []
    for change in changes:
        if change['type'] == 'update':
            result_parts.append(
                f"Document {change['doc_id']} updated:\n"
                f"Plan: {change['planned_edits']}\n"
                f"Added content: {change['value']}"
            )
        else:
            result_parts.append(
                f"New {schema_name} created:\n"
                f"Content: {change['value']}"
            )
    
    return "\n\n".join(result_parts)

# 检查 spy.called_tools 以查看提取过程中究竟发生了什么
schema_name = "Memory"
changes = extract_tool_info(spy.called_tools, schema_name)
print(changes)

## 创建 Agent

有许多不同的 Agent 架构可供选择。

这里我们将实现一个简单的 [ReAct](https://docs.langchain.com/oss/python/langgraph/workflows-agents#agents) Agent。

这个 Agent 将是一个用于创建和管理待办事项列表的有用助手。

这个 Agent 可以决定更新三种类型的长期记忆：

(a) 创建或更新带有通用用户信息的用户 `profile`

(b) 添加或更新待办事项列表 `collection` 中的条目

(c) 更新其关于如何将条目添加到待办事项列表的 `instructions`

In [ ]:
from typing import TypedDict, Literal

# 更新记忆工具
class UpdateMemory(TypedDict):
    """ 决定更新哪种记忆类型 """
    update_type: Literal['user', 'todo', 'instructions']

In [ ]:
_set_env("DEEPSEEK_API_KEY")

## 图定义

我们添加一个简单的路由器 `route_message`，做出是否保存记忆的二元决策。

和之前一样，记忆集合的更新由 `write_memory` 节点中的 `Trustcall` 处理！

In [ ]:
import uuid
from IPython.display import Image, display

from datetime import datetime
from trustcall import create_extractor
from typing import Optional
from pydantic import BaseModel, Field

from langchain_core.runnables import RunnableConfig
from langchain_core.messages import merge_message_runs, HumanMessage, SystemMessage

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, END, START
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore

from langchain_deepseek import ChatDeepSeek

# 初始化模型
model = ChatDeepSeek(model="deepseek-chat", temperature=0)

# 用户资料 schema
class Profile(BaseModel):
    """这是与你聊天的用户的资料"""
    name: Optional[str] = Field(description="The user's name", default=None)
    location: Optional[str] = Field(description="The user's location", default=None)
    job: Optional[str] = Field(description="The user's job", default=None)
    connections: list[str] = Field(
        description="Personal connection of the user, such as family members, friends, or coworkers",
        default_factory=list
    )
    interests: list[str] = Field(
        description="Interests that the user has", 
        default_factory=list
    )

# ToDo schema
class ToDo(BaseModel):
    task: str = Field(description="The task to be completed.")
    time_to_complete: Optional[int] = Field(description="Estimated time to complete the task (minutes).")
    deadline: Optional[datetime] = Field(
        description="When the task needs to be completed by (if applicable)",
        default=None
    )
    solutions: list[str] = Field(
        description="List of specific, actionable solutions (e.g., specific ideas, service providers, or concrete options relevant to completing the task)",
        min_items=1,
        default_factory=list
    )
    status: Literal["not started", "in progress", "done", "archived"] = Field(
        description="Current status of the task",
        default="not started"
    )

# 创建用于更新用户资料的 Trustcall 提取器
profile_extractor = create_extractor(
    model,
    tools=[Profile],
    tool_choice="Profile",
)

# 选择更新内容和调用什么工具的聊天机器人指令
MODEL_SYSTEM_MESSAGE = """You are a helpful chatbot. 

You are designed to be a companion to a user, helping them keep track of their ToDo list.

You have a long term memory which keeps track of three things:
1. The user's profile (general information about them) 
2. The user's ToDo list
3. General instructions for updating the ToDo list

Here is the current User Profile (may be empty if no information has been collected yet):
<user_profile>
{user_profile}
</user_profile>

Here is the current ToDo List (may be empty if no tasks have been added yet):
<todo>
{todo}
</todo>

Here are the current user-specified preferences for updating the ToDo list (may be empty if no preferences have been specified yet):
<instructions>
{instructions}
</instructions>

Here are your instructions for reasoning about the user's messages:

1. Reason carefully about the user's messages as presented below. 

2. Decide whether any of the your long-term memory should be updated:
- If personal information was provided about the user, update the user's profile by calling UpdateMemory tool with type `user`
- If tasks are mentioned, update the ToDo list by calling UpdateMemory tool with type `todo`
- If the user has specified preferences for how to update the ToDo list, update the instructions by calling UpdateMemory tool with type `instructions`

3. Tell the user that you have updated your memory, if appropriate:
- Do not tell the user you have updated the user's profile
- Tell the user them when you update the todo list
- Do not tell the user that you have updated instructions

4. Err on the side of updating the todo list. No need to ask for explicit permission.

5. Respond naturally to user user after a tool call was made to save memories, or if no tool call was made."""

# Trustcall 指令
TRUSTCALL_INSTRUCTION = """Reflect on following interaction. 

Use the provided tools to retain any necessary memories about the user. 

Use parallel tool calling to handle updates and insertions simultaneously.

System Time: {time}"""

# 更新待办事项列表的指令
CREATE_INSTRUCTIONS = """Reflect on the following interaction.

Based on this interaction, update your instructions for how to update ToDo list items. 

Use any feedback from the user to update how they like to have items added, etc.

Your current instructions are:

<current_instructions>
{current_instructions}
</current_instructions>"""

# 节点定义
def task_mAIstro(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """从存储中加载记忆，并用其个性化聊天机器人的回答。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索资料记忆
    namespace = ("profile", user_id)
    memories = store.search(namespace)
    if memories:
        user_profile = memories[0].value
    else:
        user_profile = None

    # 从存储中检索任务记忆
    namespace = ("todo", user_id)
    memories = store.search(namespace)
    todo = "\n".join(f"{mem.value}" for mem in memories)

    # 检索自定义指令
    namespace = ("instructions", user_id)
    memories = store.search(namespace)
    if memories:
        instructions = memories[0].value
    else:
        instructions = ""
    
    system_msg = MODEL_SYSTEM_MESSAGE.format(user_profile=user_profile, todo=todo, instructions=instructions)

    # 使用记忆和聊天历史进行回答
    response = model.bind_tools([UpdateMemory], parallel_tool_calls=False).invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": [response]}

def update_profile(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """反思聊天历史，并更新记忆集合。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 定义记忆的命名空间
    namespace = ("profile", user_id)

    # 检索最近的记忆作为上下文
    existing_items = store.search(namespace)

    # 为 Trustcall 提取器格式化现有记忆
    tool_name = "Profile"
    existing_memories = ([(existing_item.key, tool_name, existing_item.value)
                          for existing_item in existing_items]
                          if existing_items
                          else None
                        )

    # 合并聊天历史和指令
    TRUSTCALL_INSTRUCTION_FORMATTED=TRUSTCALL_INSTRUCTION.format(time=datetime.now().isoformat())
    updated_messages=list(merge_message_runs(messages=[SystemMessage(content=TRUSTCALL_INSTRUCTION_FORMATTED)] + state["messages"][:-1]))

    # 调用提取器
    result = profile_extractor.invoke({"messages": updated_messages, 
                                         "existing": existing_memories})

    # 将 Trustcall 产生的记忆保存到存储中
    for r, rmeta in zip(result["responses"], result["response_metadata"]):
        store.put(namespace,
                  rmeta.get("json_doc_id", str(uuid.uuid4())),
                  r.model_dump(mode="json"),
            )
    tool_calls = state['messages'][-1].tool_calls
    return {"messages": [{"role": "tool", "content": "updated profile", "tool_call_id":tool_calls[0]['id']}]}

def update_todos(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """反思聊天历史，并更新记忆集合。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]

    # 定义记忆的命名空间
    namespace = ("todo", user_id)

    # 检索最近的记忆作为上下文
    existing_items = store.search(namespace)

    # 为 Trustcall 提取器格式化现有记忆
    tool_name = "ToDo"
    existing_memories = ([(existing_item.key, tool_name, existing_item.value)
                          for existing_item in existing_items]
                          if existing_items
                          else None
                        )

    # 合并聊天历史和指令
    TRUSTCALL_INSTRUCTION_FORMATTED=TRUSTCALL_INSTRUCTION.format(time=datetime.now().isoformat())
    updated_messages=list(merge_message_runs(messages=[SystemMessage(content=TRUSTCALL_INSTRUCTION_FORMATTED)] + state["messages"][:-1]))

    # 初始化 spy 以查看 Trustcall 所做的工具调用
    spy = Spy()
    
    # 创建用于更新待办事项列表的 Trustcall 提取器
    todo_extractor = create_extractor(
    model,
    tools=[ToDo],
    tool_choice=tool_name,
    enable_inserts=True
    ).with_listeners(on_end=spy)

    # 调用提取器
    result = todo_extractor.invoke({"messages": updated_messages, 
                                    "existing": existing_memories})

    # 将 Trustcall 产生的记忆保存到存储中
    for r, rmeta in zip(result["responses"], result["response_metadata"]):
        store.put(namespace,
                  rmeta.get("json_doc_id", str(uuid.uuid4())),
                  r.model_dump(mode="json"),
            )
        
    # 响应在 task_mAIstro 中发起的工具调用，确认更新
    tool_calls = state['messages'][-1].tool_calls

    # 提取 Trustcall 所做的更改，并添加到返回给 task_mAIstro 的 ToolMessage 中
    todo_update_msg = extract_tool_info(spy.called_tools, tool_name)
    return {"messages": [{"role": "tool", "content": todo_update_msg, "tool_call_id":tool_calls[0]['id']}]}

def update_instructions(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """反思聊天历史，并更新记忆集合。"""
    
    # 从配置中获取用户 ID
    user_id = config["configurable"]["user_id"]
    
    namespace = ("instructions", user_id)

    existing_memory = store.get(namespace, "user_instructions")
        
    # 在系统提示中格式化记忆
    system_msg = CREATE_INSTRUCTIONS.format(current_instructions=existing_memory.value if existing_memory else None)
    new_memory = model.invoke([SystemMessage(content=system_msg)]+state['messages'][:-1] + [HumanMessage(content="Please update the instructions based on the conversation")])

    # 覆盖存储中的现有记忆
    key = "user_instructions"
    store.put(namespace, key, {"memory": new_memory.content})
    tool_calls = state['messages'][-1].tool_calls
    return {"messages": [{"role": "tool", "content": "updated instructions", "tool_call_id":tool_calls[0]['id']}]}

# 条件边
def route_message(state: MessagesState, config: RunnableConfig, store: BaseStore) -> Literal[END, "update_todos", "update_instructions", "update_profile"]:

    """反思记忆和聊天历史，决定是否更新记忆集合。"""
    message = state['messages'][-1]
    if len(message.tool_calls) ==0:
        return END
    else:
        tool_call = message.tool_calls[0]
        if tool_call['args']['update_type'] == "user":
            return "update_profile"
        elif tool_call['args']['update_type'] == "todo":
            return "update_todos"
        elif tool_call['args']['update_type'] == "instructions":
            return "update_instructions"
        else:
            raise ValueError

# 创建图 + 所有节点
builder = StateGraph(MessagesState)

# 定义记忆提取流程
builder.add_node(task_mAIstro)
builder.add_node(update_todos)
builder.add_node(update_profile)
builder.add_node(update_instructions)
builder.add_edge(START, "task_mAIstro")
builder.add_conditional_edges("task_mAIstro", route_message)
builder.add_edge("update_todos", "task_mAIstro")
builder.add_edge("update_profile", "task_mAIstro")
builder.add_edge("update_instructions", "task_mAIstro")

# 用于长期（跨线程）记忆的存储
across_thread_memory = InMemoryStore()

# 用于短期（线程内）记忆的检查点器
within_thread_memory = MemorySaver()

# 使用检查点器和存储编译图
graph = builder.compile(checkpointer=within_thread_memory, store=across_thread_memory)

# 查看
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
# 我们提供线程 ID 用于短期（线程内）记忆
# 我们提供用户 ID 用于长期（跨线程）记忆
config = {"configurable": {"thread_id": "1", "user_id": "Lance"}}

# 用户输入以创建资料记忆
input_messages = [HumanMessage(content="My name is Lance. I live in SF with my wife. I have a 1 year old daughter.")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用户输入待办事项
input_messages = [HumanMessage(content="My wife asked me to book swim lessons for the baby.")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用户输入以更新创建待办事项的指令
input_messages = [HumanMessage(content="When creating or updating ToDo items, include specific local businesses / vendors.")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 检查更新的指令
user_id = "Lance"

# 搜索
for memory in across_thread_memory.search(("instructions", user_id)):
    print(memory.value)

In [ ]:
# 用户输入待办事项
input_messages = [HumanMessage(content="I need to fix the jammed electric Yale lock on the door.")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用于保存记忆的命名空间
user_id = "Lance"

# 搜索
for memory in across_thread_memory.search(("todo", user_id)):
    print(memory.value)

In [ ]:
# 用户输入以更新现有待办事项
input_messages = [HumanMessage(content="For the swim lessons, I need to get that done by end of November.")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

我们可以看到 Trustcall 对现有记忆执行了修补（patching）：

https://smith.langchain.com/public/4ad3a8af-3b1e-493d-b163-3111aa3d575a/r

In [ ]:
# 用户输入待办事项
input_messages = [HumanMessage(content="Need to call back City Toyota to schedule car service.")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用于保存记忆的命名空间
user_id = "Lance"

# 搜索
for memory in across_thread_memory.search(("todo", user_id)):
    print(memory.value)

现在我们可以创建一个新线程。

这将创建一个新的会话。

保存到长期记忆的 Profile、ToDos 和 Instructions 将被访问。

In [ ]:
# 我们提供线程 ID 用于短期（线程内）记忆
# 我们提供用户 ID 用于长期（跨线程）记忆
config = {"configurable": {"thread_id": "2", "user_id": "Lance"}}

# 与聊天机器人对话
input_messages = [HumanMessage(content="I have 30 minutes, what tasks can I get done?")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 与聊天机器人对话
input_messages = [HumanMessage(content="Yes, give me some options to call for swim lessons.")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

追踪：

https://smith.langchain.com/public/84768705-be91-43e4-8a6f-f9d3cee93782/r

## Studio

![Screenshot 2024-11-04 at 1.00.19 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/6732cfb05d9709862eba4e6c_Screenshot%202024-11-11%20at%207.46.40%E2%80%AFPM.png)